# Potsdam Windowed Segmentation — Step-by-Step Tutorial

This notebook walks through every step needed to train a semantic segmentation model on the Potsdam dataset:

1. Download the data from Google Drive
2. Explore the pre-split dataset (train / validation / test)
3. Build CSV manifests for each split
4. Generate sliding-window patches from the training and validation splits
5. Inspect the patch dataset
6. Launch training

**Dataset layout** (already structured on Drive):
```
potsdam/
  train/
    images/   ← RGB GeoTIFF tiles
    masks/    ← single-band class-index GeoTIFF (0–5)
    labels/   ← RGB label tiles (for reference)
  validation/
    images/
    masks/
    labels/
  test/
    images/
    masks/
    labels/
```

**Model**: UNet + ResNet-34 backbone, 6 Potsdam classes, 256 × 256 patches

---
## Section 1 — Environment Setup

### 1.1 — Install optional download dependency

In [ ]:
# gdown is needed only for the Google Drive download step.
!pip install gdown --quiet

### 1.2 — Standard library imports

In [ ]:
import os
from pathlib import Path

### 1.3 — Scientific / geo imports

In [ ]:
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

### 1.4 — Framework imports

In [ ]:
from pytorch_segmentation_models_trainer.tools.dataset_builder.sliding_window_builder import (
    build_sliding_window_dataset,
    compute_sliding_windows,
)
from pytorch_segmentation_models_trainer.tools.inference.inference_csv_builder import (
    InferenceCSVBuilder,
)

### 1.5 — Configure root paths

**Edit `POTSDAM_ROOT` to where you want to store the data.**

In [ ]:
# ─── Edit this ───────────────────────────────────────────────────────────────
POTSDAM_ROOT = Path('/data/potsdam')
# ─────────────────────────────────────────────────────────────────────────────

SPLITS = ['train', 'validation', 'test']

# Per-split directories
IMAGES_DIRS = {s: POTSDAM_ROOT / s / 'images' for s in SPLITS}
MASKS_DIRS  = {s: POTSDAM_ROOT / s / 'masks'  for s in SPLITS}

# Output directories for windowed patches (train + validation only)
WINDOWED_ROOT = POTSDAM_ROOT / 'windowed'

POTSDAM_ROOT.mkdir(parents=True, exist_ok=True)
print('Data root:', POTSDAM_ROOT)

### 1.6 — Potsdam class definitions

In [ ]:
CLASS_NAMES = [
    'Impervious surfaces',
    'Building',
    'Low vegetation',
    'Tree',
    'Car',
    'Clutter / background',
]

# Colours from the ISPRS Potsdam label convention
CLASS_COLORS_RGB = [
    (255, 255, 255),  # 0 — Impervious surfaces
    (0,   0,   255),  # 1 — Building
    (0,   255, 255),  # 2 — Low vegetation
    (0,   255, 0  ),  # 3 — Tree
    (255, 255, 0  ),  # 4 — Car
    (255, 0,   0  ),  # 5 — Clutter
]
CLASS_COLORS_NORM = [(r/255, g/255, b/255) for r, g, b in CLASS_COLORS_RGB]
CMAP = ListedColormap(CLASS_COLORS_NORM)
NUM_CLASSES = len(CLASS_NAMES)

print(f'{NUM_CLASSES} classes:', CLASS_NAMES)

---
## Section 2 — Download the Data

The dataset is hosted on Google Drive, already split into `train / validation / test`.

### 2.1 — Download the folder from Google Drive

In [ ]:
GDRIVE_FOLDER_ID = '1n61JMwXup86z9d9tfcp7ymNaYyN6qceG'

# gdown downloads the folder contents directly into POTSDAM_ROOT.
# Skip this cell if you already have the data.
if not any((POTSDAM_ROOT / s).exists() for s in SPLITS):
    print('Downloading from Google Drive …')
    !gdown --folder {GDRIVE_FOLDER_ID} --output {POTSDAM_ROOT}
else:
    print('Data already present — skipping download.')

### 2.2 — Verify the downloaded folder structure

In [ ]:
for split in SPLITS:
    for sub in ['images', 'masks', 'labels']:
        d = POTSDAM_ROOT / split / sub
        n = len(list(d.glob('*.tif'))) if d.exists() else 0
        status = '✓' if d.exists() else '✗ MISSING'
        print(f'  {split}/{sub:<8s}: {n:>3} TIF files  {status}')

---
## Section 3 — Explore the Data

### 3.1 — Count tiles per split

In [ ]:
print(f'{'Split':<12s}  Images   Masks')
print('─' * 30)
for split in SPLITS:
    n_img  = len(list(IMAGES_DIRS[split].glob('*.tif')))
    n_mask = len(list(MASKS_DIRS[split].glob('*.tif')))
    print(f'{split:<12s}  {n_img:>6}   {n_mask:>5}')

### 3.2 — Inspect one training image (metadata)

In [ ]:
sample_img_path = sorted(IMAGES_DIRS['train'].glob('*.tif'))[0]

with rasterio.open(sample_img_path) as src:
    print('File  :', sample_img_path.name)
    print('Shape :', src.height, '×', src.width)
    print('Bands :', src.count)
    print('Dtype :', src.dtypes[0])
    print('CRS   :', src.crs)
    print('Bounds:', src.bounds)

### 3.3 — Read and display the sample image

In [ ]:
with rasterio.open(sample_img_path) as src:
    img_data = src.read()[:3]  # (3, H, W)

img_display = np.clip(img_data.transpose(1, 2, 0), 0, 255).astype(np.uint8)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(img_display)
ax.set_title(f'Train image: {sample_img_path.name}')
ax.axis('off')
plt.tight_layout()
plt.show()

### 3.4 — Inspect the corresponding mask (metadata)

In [ ]:
# Match mask by filename stem
mask_candidates = sorted(MASKS_DIRS['train'].glob('*.tif'))
sample_msk_path = next(
    (m for m in mask_candidates if m.stem == sample_img_path.stem),
    mask_candidates[0],
)

with rasterio.open(sample_msk_path) as src:
    print('File  :', sample_msk_path.name)
    print('Bands :', src.count,  '← single-band class index')
    print('Dtype :', src.dtypes[0])
    mask_data = src.read(1)  # (H, W)

unique, counts = np.unique(mask_data, return_counts=True)
print('\nClasses present:', unique)

### 3.5 — Display image and mask side by side

In [ ]:
legend_patches = [
    mpatches.Patch(color=CLASS_COLORS_NORM[i], label=f'{i} — {CLASS_NAMES[i]}')
    for i in range(NUM_CLASSES)
]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].imshow(img_display)
axes[0].set_title('RGB Image')
axes[0].axis('off')

axes[1].imshow(mask_data, cmap=CMAP, vmin=0, vmax=NUM_CLASSES - 1)
axes[1].set_title('Ground Truth Mask (single-band class index)')
axes[1].axis('off')
axes[1].legend(handles=legend_patches, loc='upper right', fontsize=9, framealpha=0.8)

plt.tight_layout()
plt.show()

### 3.6 — Class pixel distribution for the sample tile

In [ ]:
# Uses mask_data already loaded in 3.4 — no extra I/O needed.
pixel_counts = np.array([(mask_data == cls).sum() for cls in range(NUM_CLASSES)], dtype=np.int64)
total = pixel_counts.sum()

print(f'Pixel distribution in {sample_msk_path.name}:')
for i, (name, count) in enumerate(zip(CLASS_NAMES, pixel_counts)):
    print(f'  [{i}] {name:<25s}: {count:>10,}  ({100*count/total:.1f}%)')

### 3.7 — Plot class distribution bar chart (sample tile)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(
    CLASS_NAMES,
    100 * pixel_counts / total,
    color=CLASS_COLORS_NORM,
    edgecolor='grey',
    linewidth=0.5,
)
for bar, pct in zip(bars, 100 * pixel_counts / total):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f'{pct:.1f}%',
        ha='center', va='bottom', fontsize=9,
    )
ax.set_ylabel('Pixel share (%)')
ax.set_title(f'Class distribution — {sample_msk_path.name}')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

---
## Section 4 — Build Dataset Manifests

We need one CSV per split with `image` and `mask` columns.

### 4.1 — Build the train manifest

In [ ]:
train_csv = POTSDAM_ROOT / 'train.csv'

train_builder = InferenceCSVBuilder(
    images_folder=str(IMAGES_DIRS['train']),
    image_pattern='*.tif',
    recursive=False,
    masks_folder=str(MASKS_DIRS['train']),
    mask_pattern='*.tif',
    mask_suffix='',
    root_dir=str(POTSDAM_ROOT),
)
df_train = train_builder.build_csv(str(train_csv))
df_train.to_csv(train_csv, index=False)

print(f'Train manifest: {len(df_train)} rows  →  {train_csv}')

### 4.2 — Build the validation manifest

In [ ]:
val_csv = POTSDAM_ROOT / 'validation.csv'

val_builder = InferenceCSVBuilder(
    images_folder=str(IMAGES_DIRS['validation']),
    image_pattern='*.tif',
    recursive=False,
    masks_folder=str(MASKS_DIRS['validation']),
    mask_pattern='*.tif',
    mask_suffix='',
    root_dir=str(POTSDAM_ROOT),
)
df_val = val_builder.build_csv(str(val_csv))
df_val.to_csv(val_csv, index=False)

print(f'Validation manifest: {len(df_val)} rows  →  {val_csv}')

### 4.3 — Build the test manifest

In [ ]:
test_csv = POTSDAM_ROOT / 'test.csv'

test_builder = InferenceCSVBuilder(
    images_folder=str(IMAGES_DIRS['test']),
    image_pattern='*.tif',
    recursive=False,
    masks_folder=str(MASKS_DIRS['test']),
    mask_pattern='*.tif',
    mask_suffix='',
    root_dir=str(POTSDAM_ROOT),
)
df_test = test_builder.build_csv(str(test_csv))
df_test.to_csv(test_csv, index=False)

print(f'Test manifest: {len(df_test)} rows  →  {test_csv}')

### 4.4 — Preview all three manifests

In [ ]:
for name, df in [('train', df_train), ('validation', df_val), ('test', df_test)]:
    print(f'── {name} ({len(df)} rows) ──')
    display(df.head(3))

### 4.5 — Verify all mask files exist

In [ ]:
all_dfs = {'train': df_train, 'validation': df_val, 'test': df_test}

for split_name, df in all_dfs.items():
    missing = [row['mask'] for _, row in df.iterrows() if not Path(row['mask']).exists()]
    if missing:
        print(f'[{split_name}] WARNING — {len(missing)} missing masks!')
        for m in missing:
            print(f'  {m}')
    else:
        print(f'[{split_name}] All {len(df)} mask paths OK ✓')

---
## Section 5 — Build Sliding-Window Patches

We patch `train` and `validation`.  
`test` tiles are kept full-size for sliding-window inference.

### 5.1 — Configure patch parameters

In [ ]:
WINDOW_SIZE = 256    # pixels (must match backbone input size in the YAML)
OVERLAP     = 0.0    # 0 = no overlap during training patches
N_WORKERS   = 4

print(f'Patch size  : {WINDOW_SIZE} × {WINDOW_SIZE} px')
print(f'Overlap     : {OVERLAP * 100:.0f}%')
print(f'Workers     : {N_WORKERS}')
print(f'Output root : {WINDOWED_ROOT}')

### 5.2 — Estimate patch count for each split

In [ ]:
for split_name, df in [('train', df_train), ('validation', df_val)]:
    total = 0
    for _, row in df.iterrows():
        with rasterio.open(row['image']) as src:
            h, w = src.height, src.width
        n = len(compute_sliding_windows(h, w, WINDOW_SIZE, OVERLAP))
        total += n
    print(f'[{split_name}] estimated patches: {total:,}')

### 5.3 — Build windowed patches for the training split

In [ ]:
TRAIN_WINDOWED_DIR = WINDOWED_ROOT / 'train'
TRAIN_WINDOWED_DIR.mkdir(parents=True, exist_ok=True)

df_train_windowed = build_sliding_window_dataset(
    input_csv=train_csv,
    output_dir=TRAIN_WINDOWED_DIR,
    window_size=WINDOW_SIZE,
    overlap=OVERLAP,
    n_workers=N_WORKERS,
    progress=True,
)

print(f'\nTrain — {len(df_train_windowed):,} patches generated in {TRAIN_WINDOWED_DIR}')

### 5.4 — Build windowed patches for the validation split

In [ ]:
VAL_WINDOWED_DIR = WINDOWED_ROOT / 'validation'
VAL_WINDOWED_DIR.mkdir(parents=True, exist_ok=True)

df_val_windowed = build_sliding_window_dataset(
    input_csv=val_csv,
    output_dir=VAL_WINDOWED_DIR,
    window_size=WINDOW_SIZE,
    overlap=OVERLAP,
    n_workers=N_WORKERS,
    progress=True,
)

print(f'\nValidation — {len(df_val_windowed):,} patches generated in {VAL_WINDOWED_DIR}')

### 5.5 — Summary of generated patches

In [ ]:
print('Windowed patch summary:')
print(f'  train      : {len(df_train_windowed):>6,} patches')
print(f'  validation : {len(df_val_windowed):>6,} patches')
print(f'  test       : (full tiles, used for sliding-window inference)')

train_windowed_csv = TRAIN_WINDOWED_DIR / 'train.csv'
val_windowed_csv   = VAL_WINDOWED_DIR   / 'validation.csv'

print(f'\nWindowed CSVs:')
print(f'  {train_windowed_csv}')
print(f'  {val_windowed_csv}')

---
## Section 6 — Inspect the Patches

### 6.1 — Load the windowed train CSV

In [ ]:
df_patches = pd.read_csv(train_windowed_csv)
print(f'{len(df_patches):,} training patches')
df_patches.head()

### 6.2 — Sample 16 random patches

In [ ]:
N_DISPLAY   = 16
sample_rows = df_patches.sample(n=min(N_DISPLAY, len(df_patches)), random_state=42).reset_index(drop=True)
print(f'Sampled {len(sample_rows)} patches.')

### 6.3 — Read sampled patches into arrays

In [ ]:
sample_images = []
sample_masks  = []

for _, row in sample_rows.iterrows():
    with rasterio.open(row['image']) as src:
        img = np.clip(src.read()[:3].transpose(1, 2, 0), 0, 255).astype(np.uint8)
    with rasterio.open(row['mask']) as src:
        msk = src.read(1)
    sample_images.append(img)
    sample_masks.append(msk)

print(f'Loaded {len(sample_images)} pairs — image shape: {sample_images[0].shape}, mask shape: {sample_masks[0].shape}')

### 6.4 — Display image patches grid

In [ ]:
COLS = 4
ROWS = (N_DISPLAY + COLS - 1) // COLS

fig, axes = plt.subplots(ROWS, COLS, figsize=(COLS * 3, ROWS * 3))
for ax, img in zip(axes.flat, sample_images):
    ax.imshow(img)
    ax.axis('off')
for ax in list(axes.flat)[len(sample_images):]:
    ax.axis('off')

fig.suptitle('Sample training patches — images', fontsize=14)
plt.tight_layout()
plt.show()

### 6.5 — Display mask patches grid (class colours)

In [ ]:
fig, axes = plt.subplots(ROWS, COLS, figsize=(COLS * 3, ROWS * 3))
for ax, msk in zip(axes.flat, sample_masks):
    ax.imshow(msk, cmap=CMAP, vmin=0, vmax=NUM_CLASSES - 1)
    ax.axis('off')
for ax in list(axes.flat)[len(sample_masks):]:
    ax.axis('off')

legend_patches = [
    mpatches.Patch(color=CLASS_COLORS_NORM[i], label=f'{i} — {CLASS_NAMES[i]}')
    for i in range(NUM_CLASSES)
]
fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=9, framealpha=0.8)
fig.suptitle('Sample training patches — masks', fontsize=14)
plt.tight_layout(rect=[0, 0.08, 1, 0.96])
plt.show()

### 6.6 — Side-by-side comparison: image vs mask (4 pairs)

In [ ]:
N_COMPARE = 4
fig, axes = plt.subplots(2, N_COMPARE, figsize=(N_COMPARE * 3, 6))

for col in range(N_COMPARE):
    axes[0, col].imshow(sample_images[col])
    axes[0, col].set_title(f'Image #{col}')
    axes[0, col].axis('off')

    axes[1, col].imshow(sample_masks[col], cmap=CMAP, vmin=0, vmax=NUM_CLASSES - 1)
    axes[1, col].set_title(f'Mask #{col}')
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

---
## Section 7 — Training

### 7.1 — View the training configuration

In [ ]:
repo_root     = Path('..').resolve()
tutorial_yaml = (
    repo_root
    / 'pytorch_segmentation_models_trainer'
    / 'conf'
    / 'examples'
    / 'potsdam_windowed_tutorial.yaml'
)

print(f'Config: {tutorial_yaml}')
print('─' * 60)
print(tutorial_yaml.read_text())

### 7.2 — Show the exact dataset paths that will be passed

In [ ]:
train_images_dir = TRAIN_WINDOWED_DIR / 'images'
train_masks_dir  = TRAIN_WINDOWED_DIR / 'masks'
val_images_dir   = VAL_WINDOWED_DIR   / 'images'
val_masks_dir    = VAL_WINDOWED_DIR   / 'masks'

print('Paths for the training command:')
print(f'  train_dataset.image_dir : {train_images_dir}')
print(f'  train_dataset.mask_dir  : {train_masks_dir}')
print(f'  val_dataset.image_dir   : {val_images_dir}')
print(f'  val_dataset.mask_dir    : {val_masks_dir}')

### 7.3 — Build the training command

In [ ]:
train_cmd = (
    f'python -m pytorch_segmentation_models_trainer.train '
    f'--config-dir {repo_root}/pytorch_segmentation_models_trainer/conf/examples '
    f'--config-name potsdam_windowed_tutorial '
    f'train_dataset.image_dir={train_images_dir} '
    f'train_dataset.mask_dir={train_masks_dir} '
    f'val_dataset.image_dir={val_images_dir} '
    f'val_dataset.mask_dir={val_masks_dir}'
)

print('Training command:')
print()
print(train_cmd)

### 7.4 — Run training

Uncomment to run training directly from this cell.  
Alternatively, copy the command above and run it in a terminal.

In [ ]:
# import subprocess
# subprocess.run(train_cmd.split(), check=True)

### 7.5 — Monitor training with TensorBoard

In [ ]:
# Hydra writes logs to ./outputs/ by default.
# Uncomment to launch TensorBoard inline:
# %load_ext tensorboard
# %tensorboard --logdir outputs

print('Launch TensorBoard from terminal:')
print('  tensorboard --logdir outputs')

---
## Summary

| Step | Result |
|------|--------|
| Downloaded dataset | `train / validation / test` already split |
| Built manifests | `train.csv`, `validation.csv`, `test.csv` |
| Generated 256×256 patches | `windowed/train/`, `windowed/validation/` |
| Visualized data | Class distribution, patch grids, side-by-side |
| Launched training | Via Hydra config with CLI path overrides |

**Next step** → open [`02_inference_and_export.ipynb`](02_inference_and_export.ipynb)  
to run sliding-window inference on the **test** tiles and evaluate IoU.